In [1]:
import numpy as np
import pandas as pd
from ast import literal_eval

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, MultiLabelBinarizer, OneHotEncoder

In [2]:
final_df = pd.read_csv('./data/finalCards.csv')

final_df.columns

Index(['uuid', 'cardName', 'availability', 'colorIdentity', 'defense',
       'edhrecRank', 'edhrecSaltiness', 'finishes', 'isAlternative',
       'isGameChanger', 'isPromo', 'isReprint', 'isReserved', 'keywords',
       'layout', 'loyalty', 'manaCost', 'manaValue', 'cardNumber', 'power',
       'rarity', 'setCode', 'subtypes', 'supertypes', 'text', 'toughness',
       'types', 'variations', 'price', 'commander', 'setName', 'releaseDate',
       'scryfallId', 'tcgplayerProductId', 'multiverseId', 'isHuman',
       'isElemental', 'isDragon', 'isSpirit', 'isAngel', 'isElf', 'isVampire',
       'isZombie', 'isBeast', 'isWizard', 'isSoldier', 'isKnight', 'isCleric',
       'isWarrior', 'isRogue', 'isShaman', 'isDruid', 'isCreature',
       'isPlaneswalker', 'isMTGO', 'isLegal', 'isBanned', 'isFlying'],
      dtype='str')

In [3]:
creature_df = final_df.copy()
creature_df = creature_df[(creature_df['isCreature'] == 1) & (creature_df['isReprint'] == 0)]
creature_df = creature_df[['availability', 'cardName', 'colorIdentity',
       'edhrecRank', 'edhrecSaltiness', 'finishes', 'isAlternative',
       'isGameChanger', 'isPromo', 'isReserved', 'keywords',
       'layout', 'manaValue', 'power',
       'rarity', 'subtypes', 'supertypes', 'toughness',
       'types', 'price', 'commander', 'isHuman',
       'isElemental', 'isDragon', 'isSpirit', 'isAngel', 'isElf', 'isVampire',
       'isZombie', 'isBeast', 'isWizard', 'isSoldier', 'isKnight', 'isCleric',
       'isWarrior', 'isRogue', 'isShaman', 'isDruid', 'isCreature',
       'isPlaneswalker', 'isMTGO', 'isLegal', 'isBanned', 'isFlying']]

# text, cardName, scryfall ids

print(len(creature_df))
creature_df.isna().sum()

22685


availability           0
cardName               0
colorIdentity        872
edhrecRank           437
edhrecSaltiness     3923
finishes               0
isAlternative          0
isGameChanger          0
isPromo                0
isReserved             0
keywords            8733
layout                 0
manaValue              0
power                301
rarity                 0
subtypes               3
supertypes         16781
toughness            224
types                  0
price                  0
commander            565
isHuman                0
isElemental            0
isDragon               0
isSpirit               0
isAngel                0
isElf                  0
isVampire              0
isZombie               0
isBeast                0
isWizard               0
isSoldier              0
isKnight               0
isCleric               0
isWarrior              0
isRogue                0
isShaman               0
isDruid                0
isCreature             0
isPlaneswalker         0


In [4]:
# 'U' is Blue, 'B' is Black

In [5]:
creature_df.commander = creature_df.commander.fillna('NotLegal')
creature_df.commander = creature_df.commander.map({'NotLegal':0, 'Banned':0.5, 'Legal':1})

creature_df.colorIdentity = creature_df.colorIdentity.fillna('Colorless')
creature_df.colorIdentity = creature_df.colorIdentity.str.replace(',', '')

In [6]:
# how do we handle cases like these that are clearly unique
# and have "special" subtypes?
#creature_df.subtypes = creature_df.subtypes.str.replace('Lady of Proper Etiquette', 'Lady-of-Proper-Etiquette')
#creature_df.subtypes = creature_df.subtypes.str.replace('The Biggest Baddest Nastiest', 'The-Biggest-Baddest-Nastiest')

#creature_df.subtypes = creature_df.subtypes.str.replace('and/or', ',')
#creature_df.subtypes = creature_df.subtypes.str.replace(',', '')
#creature_df.subtypes = creature_df.subtypes.fillna('None')

In [7]:
"""
creature_df.keywords = creature_df.keywords.str.replace('Partner with', 'Partner-with')
creature_df.keywords = creature_df.keywords.str.replace('Hexproof from', 'Hexproof-from')
creature_df.keywords = creature_df.keywords.str.replace('Max speed', 'Max-speed')
creature_df.keywords = creature_df.keywords.str.replace('Pack tactics', 'Pack-tactics')
creature_df.keywords = creature_df.keywords.str.replace('Start your engines!', 'Start-your-engines!')
creature_df.keywords = creature_df.keywords.str.replace('More Than Meets the Eye', 'More-Than-Meets-the-Eye')
creature_df.keywords = creature_df.keywords.str.replace('Will of the Council', 'Will-of-the-Council')
creature_df.keywords = creature_df.keywords.str.replace('Open an Attraction', 'Open-an-Attraction')
creature_df.keywords = creature_df.keywords.str.replace('Roll to Visit Your Attractions', 'Roll-to-Visit-Your-Attractions')
creature_df.keywords = creature_df.keywords.str.replace('Venture into the dungeon', 'Venture-into-the-dungeon')
creature_df.keywords = creature_df.keywords.str.replace('Choose a background', 'Choose-a-background')
"""

"\ncreature_df.keywords = creature_df.keywords.str.replace('Partner with', 'Partner-with')\ncreature_df.keywords = creature_df.keywords.str.replace('Hexproof from', 'Hexproof-from')\ncreature_df.keywords = creature_df.keywords.str.replace('Max speed', 'Max-speed')\ncreature_df.keywords = creature_df.keywords.str.replace('Pack tactics', 'Pack-tactics')\ncreature_df.keywords = creature_df.keywords.str.replace('Start your engines!', 'Start-your-engines!')\ncreature_df.keywords = creature_df.keywords.str.replace('More Than Meets the Eye', 'More-Than-Meets-the-Eye')\ncreature_df.keywords = creature_df.keywords.str.replace('Will of the Council', 'Will-of-the-Council')\ncreature_df.keywords = creature_df.keywords.str.replace('Open an Attraction', 'Open-an-Attraction')\ncreature_df.keywords = creature_df.keywords.str.replace('Roll to Visit Your Attractions', 'Roll-to-Visit-Your-Attractions')\ncreature_df.keywords = creature_df.keywords.str.replace('Venture into the dungeon', 'Venture-into-the-

In [8]:
# https://mtg.wiki/page/Flavor_word
# there's way too many of these

"""
creature_df.keywords = creature_df.keywords.str.replace('Allure of Slaanesh', 'Allure-of-Slaanesh')
creature_df.keywords = creature_df.keywords.str.replace('Locus of Slaanesh', 'Locus-of-Slaanesh')
creature_df.keywords = creature_df.keywords.str.replace('Go to Sleep', 'Go-to-Sleep')
creature_df.keywords = creature_df.keywords.str.replace('How Civil of You', 'How-Civil-of-You')
creature_df.keywords = creature_df.keywords.str.replace('10000 Needles', '10000-Needles')
creature_df.keywords = creature_df.keywords.str.replace('I. AM. TALKING!', 'I.AM.TALKING!')
creature_df.keywords = creature_df.keywords.str.replace('Glory of Battle', 'Glory-of-Battle')
creature_df.keywords = creature_df.keywords.str.replace('Echo of the Lost', 'Echo-of-the-Lost')
creature_df.keywords = creature_df.keywords.str.replace('Look to the Stars', 'Look-to-the-Stars')
creature_df.keywords = creature_df.keywords.str.replace('Scavenge the Dead', 'Scavenge-the-Dead')
creature_df.keywords = creature_df.keywords.str.replace('Aegis of the Emperor', 'Aegis-of-the-Emperor')
creature_df.keywords = creature_df.keywords.str.replace('Rules of Combat', 'Rules-of-Combat')
creature_df.keywords = creature_df.keywords.str.replace('Bring it Down!', 'Bring-it-Down!')
creature_df.keywords = creature_df.keywords.str.replace('Meet in Reverse', 'Meet-in-Reverse')
creature_df.keywords = creature_df.keywords.str.replace('Terror from the Deep', 'Terror-from-the-Deep')
creature_df.keywords = creature_df.keywords.str.replace('Leading from the Front', 'Leading-from-the-Front')
creature_df.keywords = creature_df.keywords.str.replace('Hunt for Heresy', 'Hunt-for-Heresy')
creature_df.keywords = creature_df.keywords.str.replace('Hunters for Hire', 'Hunters-for-Hire')
creature_df.keywords = creature_df.keywords.str.replace('One for My Baby', 'One for My Baby')
creature_df.keywords = creature_df.keywords.str.replace('The Most Important Punch in History', 'The-Most-Important-Punch-in-History')
"""

"\ncreature_df.keywords = creature_df.keywords.str.replace('Allure of Slaanesh', 'Allure-of-Slaanesh')\ncreature_df.keywords = creature_df.keywords.str.replace('Locus of Slaanesh', 'Locus-of-Slaanesh')\ncreature_df.keywords = creature_df.keywords.str.replace('Go to Sleep', 'Go-to-Sleep')\ncreature_df.keywords = creature_df.keywords.str.replace('How Civil of You', 'How-Civil-of-You')\ncreature_df.keywords = creature_df.keywords.str.replace('10000 Needles', '10000-Needles')\ncreature_df.keywords = creature_df.keywords.str.replace('I. AM. TALKING!', 'I.AM.TALKING!')\ncreature_df.keywords = creature_df.keywords.str.replace('Glory of Battle', 'Glory-of-Battle')\ncreature_df.keywords = creature_df.keywords.str.replace('Echo of the Lost', 'Echo-of-the-Lost')\ncreature_df.keywords = creature_df.keywords.str.replace('Look to the Stars', 'Look-to-the-Stars')\ncreature_df.keywords = creature_df.keywords.str.replace('Scavenge the Dead', 'Scavenge-the-Dead')\ncreature_df.keywords = creature_df.keyw

In [9]:
#creature_df.keywords = creature_df.keywords.str.replace(',', '')
#creature_df.keywords = creature_df.keywords.fillna('None')

In [10]:
one_hot_encoding_df = creature_df.copy()
#one_hot_encoding_df['subtypes'] = creature_df['subtypes'].str.split()
#one_hot_encoding_df.subtypes

In [11]:
mlb = MultiLabelBinarizer()

# too many subtypes

#encoded_subtypes = mlb.fit_transform(one_hot_encoding_df['subtypes'])
#subtypes_df = pd.DataFrame(encoded_subtypes, columns='s_'+mlb.classes_, index=one_hot_encoding_df.index)
#subtypes_df = subtypes_df.drop(columns=['s_Advisor'])
#subtypes_df.head()

In [12]:
# one_hot_encoding_df['keywords'] = one_hot_encoding_df['keywords'].str.split()
#one_hot_encoding_df['keywords']

In [13]:
# too many keywords

#encoded_keywords = mlb.fit_transform(one_hot_encoding_df['keywords'])
#keywords_df = pd.DataFrame(encoded_keywords, columns='k_'+mlb.classes_, index=one_hot_encoding_df.index)
#keywords_df = keywords_df.drop(columns=['k_10000'])
#keywords_df.head()

In [14]:
print(len(final_df[final_df['price'] > 8]))
final_df.sort_values(by='price', ascending=False)['price']

11379


87329    75586.116667
42758    39599.535000
78287    38570.043333
74215    26250.000000
14836    20840.000000
             ...     
73087        0.030000
45825        0.030000
29011        0.030000
89391        0.020000
13008        0.020000
Name: price, Length: 94123, dtype: float64

In [15]:
one_hot_encoding_df['colorIdentity'] = one_hot_encoding_df['colorIdentity'].str.split()
one_hot_encoding_df['colorIdentity']

3              [G, R]
10                [B]
13          [B, R, U]
28                [R]
29       [B, G, U, W]
             ...     
94101          [R, U]
94103       [B, G, U]
94111             [G]
94113             [W]
94119             [G]
Name: colorIdentity, Length: 22685, dtype: object

In [16]:
encoded_colors = mlb.fit_transform(one_hot_encoding_df['colorIdentity'])
colors_df = pd.DataFrame(encoded_colors, columns='c_'+mlb.classes_, index=one_hot_encoding_df.index)
colors_df = colors_df.drop(columns=['c_B'])
colors_df.head()

,c_Colorless,c_G,c_R,c_U,c_W
3,0,1,1,0,0
10,0,0,0,0,0
13,0,0,1,1,0
28,0,0,1,0,0
29,0,1,0,1,1


In [17]:
one_hot_encoding_df['availability'].value_counts()

availability
mtgo, paper           10496
arena, mtgo, paper     8381
paper                  3808
Name: count, dtype: int64

In [18]:
one_hot_encoding_df['availability'] = one_hot_encoding_df['availability'].str.replace(',', '')
one_hot_encoding_df['availability'] = one_hot_encoding_df['availability'].str.split()
one_hot_encoding_df['availability']

3               [mtgo, paper]
10                    [paper]
13       [arena, mtgo, paper]
28              [mtgo, paper]
29       [arena, mtgo, paper]
                 ...         
94101           [mtgo, paper]
94103           [mtgo, paper]
94111    [arena, mtgo, paper]
94113           [mtgo, paper]
94119    [arena, mtgo, paper]
Name: availability, Length: 22685, dtype: object

In [19]:
encoded_availability = mlb.fit_transform(one_hot_encoding_df['availability'])
availability_df = pd.DataFrame(encoded_availability, columns='a_'+mlb.classes_, index=one_hot_encoding_df.index)
availability_df = availability_df.drop(columns=['a_mtgo'])
availability_df.head()

,a_arena,a_paper
3,0,1
10,0,1
13,1,1
28,0,1
29,1,1


In [20]:
one_hot_encoding_df['finishes'] = one_hot_encoding_df['finishes'].str.replace(',', '')
one_hot_encoding_df['finishes'] = one_hot_encoding_df['finishes'].str.split()
one_hot_encoding_df['finishes']

3        [nonfoil, foil]
10       [nonfoil, foil]
13       [nonfoil, foil]
28       [nonfoil, foil]
29       [nonfoil, foil]
              ...       
94101    [nonfoil, foil]
94103             [foil]
94111    [nonfoil, foil]
94113          [nonfoil]
94119    [nonfoil, foil]
Name: finishes, Length: 22685, dtype: object

In [21]:
encoded_finishes = mlb.fit_transform(one_hot_encoding_df['finishes'])
finishes_df = pd.DataFrame(encoded_finishes, columns='f_'+mlb.classes_, index=one_hot_encoding_df.index)
finishes_df = finishes_df.drop(columns=['f_etched'])
finishes_df.head()

,f_foil,f_nonfoil
3,1,1
10,1,1
13,1,1
28,1,1
29,1,1


In [22]:
features = pd.concat([colors_df, availability_df, finishes_df], axis=1) # keywords_df, subtypes_df
print(features.columns)
features.head()

Index(['c_Colorless', 'c_G', 'c_R', 'c_U', 'c_W', 'a_arena', 'a_paper',
       'f_foil', 'f_nonfoil'],
      dtype='str')


,c_Colorless,c_G,c_R,c_U,c_W,a_arena,a_paper,f_foil,f_nonfoil
3,0,1,1,0,0,0,1,1,1
10,0,0,0,0,0,0,1,1,1
13,0,0,1,1,0,1,1,1,1
28,0,0,1,0,0,0,1,1,1
29,0,1,0,1,1,1,1,1,1


In [23]:
X = features

y = creature_df['edhrecSaltiness']
y_mean = np.mean(y)
print(y_mean)
y = y.fillna(y_mean)

0.34593167039761213


In [24]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [25]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

knn_regressor = KNeighborsRegressor(n_neighbors=9)
knn_regressor.fit(X_train, y_train)

y_pred = knn_regressor.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'Mean Absolute Error: {mae}')
print(f'Mean Squared Error: {mse}')
print(f'R-squared: {r2}')

Mean Absolute Error: 0.1650294173250438
Mean Squared Error: 0.0588422880373553
R-squared: -0.08875639492605725


## Part 2

In [26]:
creatures_df = creature_df.copy()
creatures_df = creatures_df.join(features)
"""
creatures_df = creatures_df[['uuid', 
                            'cardName', 
                            'availability', 
                            'colorIdentity', 
                            'edhrecRank', 
                            'edhrecSaltiness', 
                            'finishes', 
                            'isAlternative', 
                            'isGameChanger',
                            'isPromo',
                            'isReserved',
                            'manaValue',
                            'cardNumber',
                            'power',
                            'rarity',
                            'setCode',
                            'subtypes',
                            'text',
                            'toughness',
                            'types',
                            'price',
                            'commander',
                            'setName',
                            'releaseDate',
                            'scryfallId',
                            'isHuman', 
                            'isElemental',
                            'isDragon', 
                            'isSpirit', 
                            'isAngel', 
                            'isElf',
                            'isVampire', 
                            'isZombie',
                            'isBeast', 
                            'isWizard', 
                            'isSoldier', 
                            'isKnight', 
                            'isCleric', 
                            'isWarrior',
                            'isRogue', 
                            'isShaman', 
                            'isDruid', 
                            'isCreature', 
                            'isMTGO', 
                            'isFlying',
                            'isLegal',
                            'isBanned',
                            'c_Colorless', 
                            'c_G', 
                            'c_R', 
                            'c_U', 
                            'c_W', 
                            'a_arena', 
                            'a_paper',
                            'f_foil', 
                            'f_nonfoil'
                            ]]
"""
print(len(creatures_df))
#creatures_df = creatures_df.dropna()
#print(len(creatures_df))

22685


In [27]:
creatures_df.isna().sum()

availability           0
cardName               0
colorIdentity          0
edhrecRank           437
edhrecSaltiness     3923
finishes               0
isAlternative          0
isGameChanger          0
isPromo                0
isReserved             0
keywords            8733
layout                 0
manaValue              0
power                301
rarity                 0
subtypes               3
supertypes         16781
toughness            224
types                  0
price                  0
commander              0
isHuman                0
isElemental            0
isDragon               0
isSpirit               0
isAngel                0
isElf                  0
isVampire              0
isZombie               0
isBeast                0
isWizard               0
isSoldier              0
isKnight               0
isCleric               0
isWarrior              0
isRogue                0
isShaman               0
isDruid                0
isCreature             0
isPlaneswalker         0


In [28]:
#creatures_df['text'] = creatures_df['text'].str.replace(r'\\n', ' ', regex=True)

In [29]:
no_var_pow_tough_all_edhrec = creatures_df.dropna(subset=['power','toughness','edhrecRank'])

In [30]:
mean_saltiness = np.mean(no_var_pow_tough_all_edhrec['edhrecSaltiness'])
print(mean_saltiness)

0.34584774151718173


In [31]:
no_var_pow_tough_all_edhrec['edhrecSaltiness'] = no_var_pow_tough_all_edhrec['edhrecSaltiness'].fillna(mean_saltiness)

In [32]:
X = no_var_pow_tough_all_edhrec[
    [
        'power', 
        'toughness', 
        'price', 
        'edhrecRank', 
        'manaValue', 
        'isHuman', 
        'isElemental',
        'isDragon', 
        'isSpirit', 
        'isAngel', 
        'isElf',
        'isVampire', 
        'isZombie',
        'isBeast', 
        'isWizard', 
        'isSoldier', 
        'isKnight', 
        'isCleric', 
        'isWarrior',
        'isRogue', 
        'isShaman', 
        'isDruid', 
        #'isCreature', 
        #'isMTGO', 
        'isFlying',
        'isLegal',
        'isBanned',
        'c_Colorless', 
        'c_G', 
        'c_R', 
        'c_U', 
        'c_W', 
        'a_arena', 
        'a_paper',
        'f_foil', 
        'f_nonfoil'
        # 'text' # remove later
        ]]

y = no_var_pow_tough_all_edhrec['edhrecSaltiness']

In [33]:
variable_toughness = creatures_df[creatures_df['toughness'].isna()]

In [34]:
variable_power = creatures_df[creatures_df['power'].isna()]

In [35]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Subtype, legality, etc.

In [36]:
#X_train = X_train.drop(columns=['text'])
#X_test = X_test.drop(columns=['text'])

In [37]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [38]:
knn_regressor = KNeighborsRegressor(n_neighbors=5)
knn_regressor.fit(X_train, y_train)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Uniform weights are used by default.See the following example for a demonstration of the impact ofdifferent weighting schemes on predictions::ref:`sphx_glr_auto_examples_neighbors_plot_regression.py`.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric metric: str, DistanceMetric object or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.If metric is a DistanceMetric object, it will be passed directly tothe underlying computation routines.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.Doesn't affect :meth:`fit` method.",None


In [39]:
y_pred = knn_regressor.predict(X_test)

In [40]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'Mean Absolute Error: {mae}')
print(f'Mean Squared Error: {mse}')
print(f'R-squared: {r2}')

Mean Absolute Error: 0.14781398817265623
Mean Squared Error: 0.04487692045274158
R-squared: 0.11423880943330156
